# Capítulo 3 · Pandas desde cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/lectures/chapter_03/02_pandas.ipynb)

**Preparación de datos para Machine Learning**

---

### ¿Por qué Pandas?

Se suele decir que un científico de datos dedica **el 80% de su tiempo a preparar datos** y solo el 20% a entrenar modelos. Los datos del mundo real llegan:
- en archivos CSV, Excel, bases de datos…
- con **valores faltantes**, duplicados y errores de escritura,
- con texto, fechas y categorías que un modelo no entiende directamente.

**Pandas** es la herramienta estándar para **cargar, explorar, limpiar y transformar** datos tabulares. Está construida sobre NumPy y se integra con Matplotlib y scikit-learn.

> 🧩 Piensa en pandas como "**Excel con superpoderes y programable**": cada tabla es un `DataFrame`, cada columna es una `Series`.

### 🎯 Objetivos
1. Dominar las dos estructuras básicas: `Series` y `DataFrame`.
2. Cargar, explorar, filtrar y transformar datos.
3. Agrupar, combinar tablas y trabajar con fechas.

### 📚 Índice
1. `Series`
2. `DataFrame`
3. Cargar datos: nuestro dataset de viviendas
4. Explorar un dataset
5. Seleccionar y filtrar
6. Crear y modificar columnas
7. Agrupar y resumir (`groupby`, `pivot_table`)
8. Combinar tablas (`merge`, `concat`)
9. Fechas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Versión de pandas:", pd.__version__)

> 💡 Convención universal: `import pandas as pd`.

---
## 1. `Series`: una columna con etiquetas

Una `Series` es un array de NumPy de 1 dimensión **con un índice** (etiquetas para cada valor).

In [ ]:
temperaturas = pd.Series([22.5, 25.1, 19.8, 30.2],
                         index=["lunes", "martes", "miércoles", "jueves"],
                         name="temperatura")
temperaturas

In [ ]:
print(temperaturas["martes"])        # acceder por etiqueta
print(temperaturas.values)           # el array de NumPy que hay debajo
print(temperaturas.index)

In [ ]:
# Las operaciones vectorizadas funcionan igual que en NumPy
print(temperaturas * 9 / 5 + 32)     # a Fahrenheit
print(temperaturas[temperaturas > 22])
print("Media:", temperaturas.mean())

In [ ]:
# value_counts: ¿cuántas veces aparece cada valor? (MUY usado con variables categóricas)
colores = pd.Series(["rojo", "azul", "rojo", "verde", "rojo", "azul"])
colores.value_counts()

---
## 2. `DataFrame`: una tabla

Un `DataFrame` es una tabla de 2 dimensiones: **filas** (con un índice) y **columnas** (cada una es una `Series` y puede tener un tipo distinto).

In [ ]:
# Crear un DataFrame a partir de un diccionario: clave = nombre de columna
df = pd.DataFrame({
    "nombre": ["Ana", "Luis", "Marta", "Pedro", "Lucía"],
    "edad": [28, 35, 22, 41, 30],
    "ciudad": ["Madrid", "Lima", "Bogotá", "Lima", "Madrid"],
    "salario": [32000, 45000, 28000, 52000, 39000],
})
df

In [ ]:
print("Forma:", df.shape)            # (filas, columnas)
print("Columnas:", list(df.columns))
print("Índice:", df.index)
print()
print(df.dtypes)                     # tipo de cada columna

In [ ]:
# También desde un array de NumPy (lo típico al salir de un modelo)
matriz = np.random.default_rng(0).normal(size=(4, 3))
pd.DataFrame(matriz, columns=["feature_1", "feature_2", "feature_3"])

In [ ]:
# Y de vuelta a NumPy (lo que necesita un modelo de ML)
df[["edad", "salario"]].to_numpy()

---
## 3. Cargar datos: nuestro dataset de viviendas 🏠

Trabajaremos con un dataset de ventas de viviendas que está en `data/viviendas.csv`. Si todavía no tienes los datos del curso, descárgalos desde la raíz del repo:

```bash
uv run python scripts/download_data.py
```

In [ ]:
# Los datos del curso están en la carpeta data/, en la raíz del repo.
# Para encontrar la raíz, subimos de carpeta en carpeta hasta dar con pyproject.toml.
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

Cargamos el CSV con `pd.read_csv`, como harías con cualquier dataset real:

In [ ]:
df = pd.read_csv(DATA_DIR / "viviendas.csv")
df

> 📝 **Otras formas de cargar datos** (para que las conozcas):
> ```python
> pd.read_csv("https://url/a/un/archivo.csv")      # directamente desde internet
> pd.read_csv("archivo.csv", sep=";", decimal=",")  # CSV con formato europeo/latino
> pd.read_excel("archivo.xlsx", sheet_name="Hoja1") # Excel (requiere openpyxl)
> pd.read_json("archivo.json")
> pd.read_parquet("archivo.parquet")                # formato muy eficiente para big data
> pd.read_sql("SELECT * FROM tabla", conexion)      # bases de datos
> ```

---
## 4. Explorar un dataset

Lo **primero** que se hace con cualquier dataset nuevo. Estas funciones son tu "radiografía" de los datos:

In [ ]:
df.head()          # primeras 5 filas (df.head(10) para 10)

In [ ]:
df.tail(3)         # últimas filas

In [ ]:
df.sample(5, random_state=0)    # filas aleatorias (a veces el inicio no es representativo)

In [ ]:
df.info()          # columnas, tipos, y ¡cuántos valores NO nulos hay!

> 🔍 Fíjate en `info()`: hay **600 filas** y ninguna columna tiene valores nulos. Pero `fecha_venta` aparece como texto (`str`), no como fecha: la convertiremos en la sección de fechas.

In [ ]:
df.describe()      # estadísticas de las columnas numéricas

> 🔍 Con `describe()` ves de un vistazo la escala de cada variable: cuántos valores hay (`count`), la media, la dispersión (`std`) y el rango (`min`, `max`).

In [ ]:
df.describe(exclude="number")     # resumen de las columnas no numéricas

In [ ]:
df["barrio"].value_counts()

In [ ]:
df.nunique()       # número de valores distintos por columna

---
## 5. Seleccionar y filtrar

### 5.1 Seleccionar columnas

In [ ]:
df["precio"].head()                    # una columna → Series

In [ ]:
df[["barrio", "metros", "precio"]].head()   # varias columnas → DataFrame (¡doble corchete!)

### 5.2 Seleccionar filas: `loc` e `iloc` ⭐

| | `loc` | `iloc` |
|---|---|---|
| Selecciona por… | **etiquetas** (nombres del índice y de columnas) | **posiciones** (números enteros, como en NumPy) |
| Ejemplo | `df.loc[5, "precio"]` | `df.iloc[5, 7]` |
| Slicing | incluye el final | excluye el final |

In [ ]:
df.iloc[0]                 # primera fila (por posición)

In [ ]:
df.iloc[:3, :4]            # 3 primeras filas, 4 primeras columnas

In [ ]:
df.loc[:4, ["barrio", "precio"]]     # filas con índice 0 a 4 (¡incluido!) y columnas por nombre

### 5.3 Filtrar con condiciones (máscaras booleanas) ⭐

Exactamente igual que en NumPy:

In [ ]:
df[df["precio"] > 400_000].head()

In [ ]:
# Varias condiciones: & (y), | (o), ~ (no). ¡Paréntesis obligatorios!
grandes_con_garaje = df[(df["metros"] > 150) & (df["garaje"] == "sí")]
print(len(grandes_con_garaje), "viviendas grandes con garaje")
grandes_con_garaje.head()

In [ ]:
# isin: ¿el valor está en una lista?
df[df["barrio"].isin(["Norte", "Sur"])].head()

In [ ]:
# between: rango de valores (incluye los extremos)
df[df["antiguedad"].between(0, 5)].shape

In [ ]:
# query: una alternativa más legible para filtros complejos
df.query("habitaciones >= 4 and precio < 300000").head()

In [ ]:
# Filtrar Y seleccionar columnas a la vez con loc
df.loc[df["barrio"] == "Centro", ["metros", "precio"]].head()

### 5.4 Ordenar

In [ ]:
df.sort_values("precio", ascending=False).head()

In [ ]:
df.sort_values(["barrio", "precio"], ascending=[True, False]).head()

---
## 6. Crear y modificar columnas

In [ ]:
# Para no estropear los datos originales, trabajamos sobre una copia
ejemplo = df.head(8).copy()

# Nueva columna a partir de otras (operación vectorizada)
ejemplo["precio_m2"] = ejemplo["precio"] / ejemplo["metros"]

# Columna condicional con np.where
ejemplo["es_nueva"] = np.where(ejemplo["antiguedad"] < 10, "nueva", "usada")

ejemplo[["metros", "precio", "precio_m2", "antiguedad", "es_nueva"]]

In [ ]:
# map: sustituir valores usando un diccionario (ideal para codificar categorías)
ejemplo["garaje_num"] = ejemplo["garaje"].map({"sí": 1, "no": 0})

# apply: aplicar una función propia a cada valor
def tamaño(m):
    if m < 60:
        return "pequeña"
    elif m < 120:
        return "mediana"
    return "grande"

ejemplo["tamaño"] = ejemplo["metros"].apply(tamaño)
ejemplo[["garaje", "garaje_num", "metros", "tamaño"]]

> ⚡ **Rendimiento**: `apply` es en realidad un bucle de Python, así que es lento con millones de filas. Siempre que puedas, usa operaciones vectorizadas (`+`, `*`, `np.where`, `.str`, `.dt`, `pd.cut`…).

In [ ]:
# pd.cut: convertir una variable numérica en categorías por rangos (binning) — ¡vectorizado!
ejemplo["tamaño_v2"] = pd.cut(ejemplo["metros"], bins=[0, 60, 120, np.inf],
                              labels=["pequeña", "mediana", "grande"])
ejemplo[["metros", "tamaño", "tamaño_v2"]]

In [ ]:
# Renombrar y eliminar columnas
ejemplo = ejemplo.rename(columns={"metros": "superficie_m2"})
ejemplo = ejemplo.drop(columns=["tamaño_v2", "es_nueva"])
ejemplo.columns

> ⚠️ **Buena práctica**: la mayoría de métodos de pandas **devuelven un DataFrame nuevo** y no modifican el original. Por eso escribimos `df = df.metodo(...)`. Para modificar valores concretos usa siempre `df.loc[filas, columna] = valor`.

---
## 7. Agrupar y resumir ⭐

`groupby` sigue el patrón **dividir → aplicar → combinar**: separa las filas en grupos, calcula algo en cada grupo y junta los resultados.

In [ ]:
df.groupby("barrio")["precio"].mean().sort_values(ascending=False)

In [ ]:
# Varias estadísticas a la vez con agg
df.groupby("barrio")["precio"].agg(["count", "mean", "median", "min", "max"]).round(0)

In [ ]:
# Estadísticas distintas para columnas distintas, con nombres personalizados
resumen = df.groupby("barrio").agg(
    n_ventas=("id", "count"),
    precio_medio=("precio", "mean"),
    metros_medios=("metros", "mean"),
    pct_garaje=("garaje", lambda s: (s == "sí").mean() * 100),
).round(1)
resumen

In [ ]:
# Agrupar por varias columnas
df.groupby(["barrio", "garaje"])["precio"].mean().round(0).unstack()

### Tablas dinámicas y tablas cruzadas

In [ ]:
# pivot_table: como las tablas dinámicas de Excel
pd.pivot_table(df, values="precio", index="barrio", columns="habitaciones",
               aggfunc="mean").round(-3)

In [ ]:
# crosstab: frecuencias cruzadas entre dos variables categóricas
pd.crosstab(df["barrio"], df["garaje"], normalize="index").round(2)   # proporciones por fila

---
## 8. Combinar tablas

Los datos suelen estar repartidos en varias tablas. Imagina que tenemos otra tabla con información de cada barrio:

In [ ]:
info_barrios = pd.DataFrame({
    "barrio": ["Centro", "Norte", "Sur", "Este", "Oeste", "Puerto"],
    "distancia_centro_km": [0.0, 5.5, 7.0, 4.0, 6.0, 12.0],
    "indice_seguridad": [7.2, 8.5, 6.1, 7.8, 8.0, 6.5],
})
info_barrios

### `merge`: unir por una columna en común (como un `JOIN` de SQL)

| `how=` | Resultado |
|---|---|
| `"inner"` | solo las filas cuya clave existe en **ambas** tablas |
| `"left"` | todas las filas de la tabla izquierda (lo más habitual) |
| `"right"` | todas las de la derecha |
| `"outer"` | todas las filas de ambas |

In [ ]:
df = df.merge(info_barrios, on="barrio", how="left")
df[["barrio", "precio", "distancia_centro_km", "indice_seguridad"]].head()

> 💡 "Puerto" no aparece en nuestras ventas, así que con `how="left"` simplemente se ignora.

### `concat`: apilar tablas

In [ ]:
ventas_enero = pd.DataFrame({"producto": ["A", "B"], "unidades": [10, 5]})
ventas_febrero = pd.DataFrame({"producto": ["A", "C"], "unidades": [7, 12]})

pd.concat([ventas_enero, ventas_febrero], ignore_index=True)

---
## 9. Fechas

Al cargar el CSV, `fecha_venta` llegó como texto. Primero la convertimos a fecha con `pd.to_datetime`:

In [ ]:
df["fecha_venta"] = pd.to_datetime(df["fecha_venta"])
df.dtypes

Con la columna convertida a `datetime`, el accesor `.dt` nos da acceso a sus componentes:

In [ ]:
df["año"] = df["fecha_venta"].dt.year
df["mes"] = df["fecha_venta"].dt.month
df["dia_semana"] = df["fecha_venta"].dt.dayofweek      # 0 = lunes, 6 = domingo
df[["fecha_venta", "año", "mes", "dia_semana"]].head()

In [ ]:
# Filtrar por fechas
df[df["fecha_venta"] >= "2024-06-01"].shape

In [ ]:
# Serie temporal: precio medio por mes ("ME" = fin de mes)
precio_mensual = df.set_index("fecha_venta")["precio"].resample("ME").mean()
precio_mensual.head()